# CLIMETLAB

In [1]:
#! pip install -U climetlab --quiet
#! pip install -U climetlab_s2s_ai_challenge --quiet

%cd -q ..

In [2]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Flatten

In [3]:
import climetlab as cml

In [4]:
highlow = cml.load_dataset("high-low")
for field, label in highlow.fields():
    cml.plot_map(field, width=256, title=highlow.title(label))

2024-02-16 12:16:04,258 INFO Welcome to the CDS


2024-02-16 12:16:04,258 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/resources/reanalysis-era5-pressure-levels


2024-02-16 12:16:04,511 INFO Request is queued


KeyboardInterrupt: 

In [ ]:
xy, xy_test = highlow.load_data(
    test_size=0.3
)
xy_train = [i[:20]for i in xy]
xy_valid = [i[20:]for i in xy]
n_classes = 4

In [ ]:
from crest.data.loading.StructuredDataset import StructuredDataset

INP = 'pressure'
# INP2 = 'metadata'
OUT = 'class'

train_ds = StructuredDataset(*xy_train, labels=[INP, OUT])
valid_ds = StructuredDataset(*xy_valid, labels=[INP, OUT])
test_ds = StructuredDataset(*xy_test, labels=[INP, OUT])

In [ ]:
from crest.data.Batcher import Batcher

batch_kwargs = {
    'batch_size': 1,
    'features' : [[INP], [OUT]],
    'repeat' : True,
    'workers' : 1
}

train_batcher = Batcher(train_ds, **batch_kwargs)
batch_kwargs['shuffle'] = False
valid_batcher = Batcher(valid_ds, **batch_kwargs)
test_batcher = Batcher(test_ds, **batch_kwargs)

In [ ]:
from crest.model.HierarchalTensorGraph import HierarchalTensorGraph as HTG

def crest_layer(layer):
    graph = HTG(lambda d: {OUT: layer(d[INP])}, name=layer.name)
    graph.inputs = {INP: None}
    graph.outputs = {OUT: None}
    return graph

layers = [crest_layer(layer) for layer in [
    Flatten(),
    Dense(64, activation="sigmoid"),
    Dense(n_classes, activation="softmax")
]]

In [ ]:
for i in layers[:-1]:
    i.rename_io(outputs_map={OUT: INP})

graph = HTG(
    name='ClimetPressure',
    inputs={INP: tf.TensorSpec(shape=[None, 21, 21])},
    outputs={OUT: tf.TensorSpec(shape=[None, n_classes])},
)

print(list(map(graph.add_edge, ['input']+layers, layers+['output'])))

In [ ]:
from crest.model.Model import Model 

build_kwargs = {
    'optimizer': 'Adam',
    'loss' : 'categorical_crossentropy',
    'metrics' : 'accuracy' 
}

model = Model(graph)
model.build(**build_kwargs)
model.fit(train_batcher, **{
    'validation_data' : valid_batcher,
    'steps_per_epoch' : 5,
    'validation_steps' : 1,
    'epochs' : 100,
    'verbose' : 1
})

In [ ]:
accuracy = model.evaluate(test_batcher,verbose=False,return_dict=True)
print(f'Test performance metrics: {accuracy}')

In [ ]:
train_batcher.close()
test_batcher.close()
valid_batcher.close()

In [ ]:
x_pred,_ = xy_test  # Use images from the testing set
ds_pred = StructuredDataset(*[x_pred], labels=[INP])
batch_pred = Batcher(ds_pred, **{
    'batch_size' : 5,
    'features'   : [INP],
    'repeat'     : True,
    'workers'    : 0,
})

In [ ]:
pred_kwargs = {
    'batch_size' : 5,
    'verbose' : True,
    'steps' : 1
}

In [ ]:
model_pred_batch = model.predict(batch_pred, **pred_kwargs) # Using a batcher
model_pred_dataset = model.predict(ds_pred,**pred_kwargs)   # Using a dataset
print('Difference:', (model_pred_batch['class'] - model_pred_dataset['class']).sum())

In [ ]:
import src.data.archiver as archiver

arch = archiver.Archiver()
for batch in train_batcher: 
    model.predict(batch, **pred_kwargs)
    arch.archive(model)

arch.close()

In [ ]:
(x_train, y_train, f_train), (x_test, y_test, f_test) = highlow.load_data(
    test_size=0.3,
    fields=True
)


for p, f in zip(model_pred_batch['class'], f_test):
    cml.plot_map(f, width=256, title= highlow.title(p))

In [ ]:
x_train

In [ ]:
y_train

In [ ]:
f_train